In [2]:
import os
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
import torch
import timm
from torchvision.io import read_image, ImageReadMode
import torch.nn.functional as F
import numpy as np
import torch.optim as optim
from torch import nn
from collections import Counter
from torchvision.transforms import v2


c:\Users\guilh\Git\Elifoot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def split_data(data_list, train_proportion=0.7, test_proportion=0.2):
    data_list_len = len(data_list)
    shuffled_data = list(np.random.permutation(data_list))

    train_size = int(np.ceil(data_list_len * train_proportion))
    test_size = int(np.ceil(data_list_len * test_proportion))
    
    return {'train': shuffled_data[0: train_size], 
            'test': shuffled_data[train_size: train_size + test_size], 
            'val': shuffled_data[train_size + test_size:]}


In [4]:
class EliImageDataset(Dataset):
    def __init__(self, img_path_list, all_classes_df=None, transform=None):

        self.img_path_list = img_path_list
        self.img_path_df = pd.Series(self.img_path_list).explode().reset_index()
        self.img_path_df.columns = ['Label', 'Img_path'] 

        if all_classes_df is not None:
            self.all_classes_df = all_classes_df
        else:
            cd = Counter([i.split('\\')[-2] for i in self.img_path_list])
            classes_df = pd.DataFrame(cd, index=[0]).transpose().reset_index()
            classes_df.columns = ['Class', 'Count']
            classes_df['Label'] = classes_df.index
            self.all_classes_df = classes_df

        if not transform:
            self.transform = v2.Compose([
                v2.ToImage(),             
                v2.ToDtype(torch.float32, scale=True),
                v2.Resize((120, 120), antialias=True)
                ])
        else:
            self.transform = transform


    def __len__(self):
        return len(self.img_path_list)

    def __getitem__(self, idx):
        image = read_image(self.img_path_df.iloc[idx]['Img_path'], mode=ImageReadMode.RGB)
        label = self.img_path_df.iloc[idx]['Img_path'].split('\\')[2]

        if self.transform:
            image = self.transform(image)

        label = self.all_classes_df.loc[self.all_classes_df['Class'] == label, 'Label'].item()
        return image, label

In [5]:
img_labels_folder = r'Data\Frames_Categories'
img_path_dict = {}
for folder in os.listdir(img_labels_folder):
    if folder == 'Other':
        continue
    folder_path = os.path.join(img_labels_folder, folder)
    img_path_dict[folder] =[os.path.join(folder_path, img_name) for img_name in  os.listdir(folder_path)]

In [6]:
imgs_split_dict = {split_type: [] for split_type in ['train', 'test', 'val']}

for img_class in img_path_dict.keys():
    if len(img_path_dict[img_class]) < 1000:
        continue
    class_split_dict = split_data(img_path_dict[img_class])
    for split_type in class_split_dict.keys(): 
        imgs_split_dict[split_type] += class_split_dict[split_type]

In [7]:
transforms = v2.Compose([
    v2.RandomHorizontalFlip(p=0.5),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    v2.RandomResizedCrop(size=(120, 120), antialias=True),
])

In [8]:
ds_train = EliImageDataset(imgs_split_dict['train'], transform=transforms)
ds_val = EliImageDataset(imgs_split_dict['val'], ds_train.all_classes_df)
ds_test = EliImageDataset(imgs_split_dict['test'], ds_train.all_classes_df)

In [9]:
train_dataloader = DataLoader(ds_train, batch_size=64, shuffle=True)
val_dataloader = DataLoader(ds_val, batch_size=64)
test_dataloader = DataLoader(ds_test, batch_size=64)

In [10]:
1/0

ZeroDivisionError: division by zero

In [11]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [12]:
model = timm.create_model('convnext_tiny', pretrained=True, num_classes=ds_train.all_classes_df.shape[0])

In [13]:
w = torch.tensor(ds_train.all_classes_df['Count'] / ds_train.all_classes_df['Count'].sum()).float()

In [36]:
model = timm.create_model('convnext_tiny', pretrained=True, num_classes=ds_train.all_classes_df.shape[0])

criterion = torch.nn.CrossEntropyLoss()#1/w
optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer, mode='max',
                                                       patience=1, factor=0.4)
softmax = nn.Softmax(dim=1)

In [ ]:
all_losses = []
d = {}
train_outputs_d = {}

for epoch in range(50):
    epoch_train_loss = []
    epoch_preds = []
    epoch_labels = []
    train_outputs_d[epoch] = {}
    counter = 0
    for img, labels in train_dataloader:

        outputs = model.to(device)(img.to(device))
        loss = criterion(outputs, labels.to(device))

        preds = softmax(outputs).max(axis=1).indices.tolist()
        epoch_train_loss += [loss.item()]

        optimizer.zero_grad()


        loss.backward()
        optimizer.step()

        epoch_outputs = preds
        epoch_labels += [label.item() for label in labels]
        #Accuracy

        train_outputs_d[epoch][counter] = outputs
        counter += 1
    
    epoch_mean_loss_train = np.mean(epoch_train_loss)
    d[epoch] = pd.DataFrame([epoch_preds, epoch_labels]).transpose()
 
    all_preds_val = []
    all_labels_val = []

    for img_val, label_val in test_dataloader:
        with torch.no_grad():
            outputs = model.to(device)(img_val)
            all_preds_val += softmax(outputs).max(axis=1).indices.tolist()
            all_labels_val += label_val.tolist()

    preds_df = pd.DataFrame([all_preds_val, all_labels_val]).transpose()
    preds_df.columns = ['Pred', 'Label']
    
    epoch_mean_loss = np.mean(preds_df['Pred'] == preds_df['Label'])
    print(f'Epoch {epoch}, Train:{epoch_mean_loss_train}, Val accuracy:{epoch_mean_loss}')
 


Epoch 0, Train:0.17084489940580996, Val accuracy:0.9556527170518426
Epoch 1, Train:0.10136466119861738, Val accuracy:0.9531542785758901
Epoch 2, Train:0.09055368128147992, Val accuracy:0.9681449094316052


In [ ]:
# Move model to device ONCE
model = timm.create_model('convnext_tiny', pretrained=True, num_classes=ds_train.all_classes_df.shape[0])
model = model.to(device)

criterion = torch.nn.CrossEntropyLoss()
# CRITICAL FIX: Lower learning rate for pretrained model
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)  
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer, mode='max',
                                                       patience=1, factor=0.4)
softmax = nn.Softmax(dim=1)

all_losses = []
d = {}
train_outputs_d = {}

for epoch in range(50):
    model.train()  # Set to training mode
    epoch_train_loss = []
    epoch_preds = []
    epoch_labels = []
    train_outputs_d[epoch] = {}
    counter = 0
    
    for img, labels in train_dataloader:
        img = img.to(device)
        labels = labels.to(device)
        
        outputs = model(img)  # Model already on device
        loss = criterion(outputs, labels)
        
        preds = outputs.argmax(dim=1).cpu().tolist()  # Fixed prediction
        epoch_preds += preds  # FIX: Actually append predictions
        epoch_labels += labels.cpu().tolist()
        epoch_train_loss.append(loss.item())
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_outputs_d[epoch][counter] = outputs.detach().cpu()
        counter += 1
    
    # Calculate training accuracy
    train_acc = np.mean(np.array(epoch_preds) == np.array(epoch_labels))
    epoch_mean_loss_train = np.mean(epoch_train_loss)
    
    # Validation
    model.eval()  # Set to eval mode
    all_preds_val = []
    all_labels_val = []
    
    for img_val, label_val in test_dataloader:
        img_val = img_val.to(device)  # FIX: Move to device
        
        with torch.no_grad():
            outputs = model(img_val)
            all_preds_val += outputs.argmax(dim=1).cpu().tolist()
            all_labels_val += label_val.tolist()
    
    val_acc = np.mean(np.array(all_preds_val) == np.array(all_labels_val))
    
    print(f'Epoch {epoch}, Train Loss: {epoch_mean_loss_train:.4f}, Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}')
    
    # Update scheduler based on validation accuracy
    scheduler.step(val_acc)
    
    d[epoch] = pd.DataFrame({'Pred': epoch_preds, 'Label': epoch_labels})

Epoch 0, Train Loss: 0.1454, Train Acc: 0.9415, Val Acc: 0.9382
Epoch 1, Train Loss: 0.0920, Train Acc: 0.9613, Val Acc: 0.9700
Epoch 2, Train Loss: 0.0963, Train Acc: 0.9604, Val Acc: 0.9588
Epoch 3, Train Loss: 0.0840, Train Acc: 0.9648, Val Acc: 0.9600
Epoch 4, Train Loss: 0.0570, Train Acc: 0.9781, Val Acc: 0.9725
Epoch 5, Train Loss: 0.0538, Train Acc: 0.9782, Val Acc: 0.9694
Epoch 6, Train Loss: 0.0451, Train Acc: 0.9813, Val Acc: 0.9644
Epoch 7, Train Loss: 0.0430, Train Acc: 0.9827, Val Acc: 0.9694
Epoch 8, Train Loss: 0.0364, Train Acc: 0.9854, Val Acc: 0.9594
Epoch 9, Train Loss: 0.0354, Train Acc: 0.9863, Val Acc: 0.9731
Epoch 10, Train Loss: 0.0248, Train Acc: 0.9909, Val Acc: 0.9719
Epoch 11, Train Loss: 0.0282, Train Acc: 0.9882, Val Acc: 0.9769
Epoch 12, Train Loss: 0.0284, Train Acc: 0.9891, Val Acc: 0.9738
Epoch 13, Train Loss: 0.0272, Train Acc: 0.9884, Val Acc: 0.9744
Epoch 14, Train Loss: 0.0240, Train Acc: 0.9907, Val Acc: 0.9731
Epoch 15, Train Loss: 0.0255, Train

KeyboardInterrupt: 

In [ ]:
softmax(train_outputs_d[i][0][0:2])

tensor([[0.1111, 0.0381, 0.1184, 0.1745, 0.0959, 0.2267, 0.2353],
        [0.3568, 0.0547, 0.0210, 0.1778, 0.0628, 0.1460, 0.1809]],
       grad_fn=<SoftmaxBackward0>)

In [ ]:
for i in range(10):
    print(softmax(train_outputs_d[i][0][0:2]).max(axis=1))
    print('-----------------------------------------------')

torch.return_types.max(
values=tensor([0.2353, 0.3568], grad_fn=<MaxBackward0>),
indices=tensor([6, 0]))
-----------------------------------------------
torch.return_types.max(
values=tensor([0.9250, 0.9250], grad_fn=<MaxBackward0>),
indices=tensor([2, 2]))
-----------------------------------------------
torch.return_types.max(
values=tensor([0.5980, 0.5980], grad_fn=<MaxBackward0>),
indices=tensor([0, 0]))
-----------------------------------------------
torch.return_types.max(
values=tensor([0.9805, 0.9805], grad_fn=<MaxBackward0>),
indices=tensor([5, 5]))
-----------------------------------------------
torch.return_types.max(
values=tensor([0.5222, 0.5222], grad_fn=<MaxBackward0>),
indices=tensor([3, 3]))
-----------------------------------------------
torch.return_types.max(
values=tensor([0.7395, 0.7395], grad_fn=<MaxBackward0>),
indices=tensor([6, 6]))
-----------------------------------------------
torch.return_types.max(
values=tensor([0.7226, 0.7226], grad_fn=<MaxBackward0>),
i

In [ ]:
for i in d.keys():
    print(i, np.mean(d[i].iloc[:,0] == d[i].iloc[:,1]))

0 0.0
1 0.0
2 0.0
3 0.0
4 0.0
5 0.0
6 0.0
7 0.0
8 0.0
9 0.0
10 0.0
11 0.0
12 0.0
13 0.0
14 0.0
15 0.0
16 0.0
17 0.0
18 0.0
19 0.0
20 0.0
21 0.0
22 0.0
23 0.0
24 0.0
25 0.0
26 0.0
27 0.0
28 0.0
29 0.0
30 0.0
31 0.0
32 0.0
33 0.0
34 0.0
35 0.0
36 0.0
37 0.0
38 0.0
39 0.0
40 0.0
41 0.0
42 0.0
43 0.0
44 0.0
45 0.0
46 0.0
47 0.0
48 0.0
49 0.0


In [ ]:
for i in d.keys():
    print(set(d[i]))

{0, 1}


In [ ]:
ds_test = EliImageDataset(imgs_split_dict['train'])
test_dataloader = DataLoader(ds_test, batch_size=64, shuffle=False)

In [ ]:
all_preds = []
all_labels = []

for img, labels in test_dataloader:
     with torch.no_grad():
          outputs = model.to(device)(img)
          all_preds += softmax(outputs).max(axis=1).indices.tolist()
          all_labels += labels.tolist()

In [ ]:
os.listdir('Data/Frames_Categories/dogs')[0].replace('.jpg', '.jpeg')

'dog.1.jpeg'